In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pathlib import Path

ROOT = Path(
    "/content/drive/MyDrive/Deepfake_Dissertation"
)

folders = [
    "raw",
    "pilot/ro",
    "pilot/en",
    "metadata",
    "results"
]

for folder in folders:
    (ROOT / folder).mkdir(
        parents=True,
        exist_ok=True
    )

print("Project folders created!")

Project folders created!


In [ ]:
!pip -q install pandas mutagen tqdm

In [ ]:
import tarfile

archives = list((ROOT / "raw").glob("*.tar.gz"))

print("Archives found:")
for archive in archives:
    print(archive.name)

Archives found:
1789492666139-cv-corpus-27.0-2026-09-11-ro.tar.gz


In [ ]:
import shutil
import subprocess

archive_path = archives[0]

local_archive = Path("/content/commonvoice_ro.tar.gz")
extract_dir = Path("/content/commonvoice_ro")

extract_dir.mkdir(exist_ok=True)

shutil.copy2(archive_path, local_archive)

subprocess.run(
    ["tar", "-xzf", str(local_archive),
     "-C", str(extract_dir)],
    check=True
)

print("Extraction complete!")

Extraction complete!


In [ ]:
metadata_files = list(
    extract_dir.rglob("validated.tsv")
)

print(metadata_files)

[PosixPath('/content/commonvoice_ro/cv-corpus-27.0-2026-09-11/ro/validated.tsv')]


In [ ]:
import pandas as pd

metadata_path = metadata_files[0]

df = pd.read_csv(
    metadata_path,
    sep="\t",
    dtype=str
)

print("Total validated recordings:", len(df))
print(df.columns.tolist())

display(df.head())

Total validated recordings: 22468
['client_id', 'path', 'sentence_id', 'sentence', 'sentence_domain', 'up_votes', 'down_votes', 'age', 'gender', 'accents', 'variant', 'locale', 'segment']


,client_id,path,sentence_id,sentence,sentence_domain,up_votes,down_votes,age,gender,accents,variant,locale,segment
0,01c303123c6fc48bdb14ea51040a968a8be49185ab8cb1...,common_voice_ro_37782249.mp3,ceffa8d3188ffd2d1811f8f25b564cb41559f1c60aa196...,Datoriile și deficitele nu sunt decât pretexte.,NaN,2,0,fifties,female_feminine,NaN,NaN,ro,NaN
1,05e843fe49983ae209678d47b0de40b4ed64675d2b84c5...,common_voice_ro_27304814.mp3,29bbe5fc1729cb6648fecf1b9a6d052f6189205f9f2f4e...,Eu am votat în favoarea acestui raport.,NaN,2,0,NaN,NaN,NaN,NaN,ro,NaN
2,06f91a48232da34bbd26b9b270591c2e295e9522dd8839...,common_voice_ro_40018473.mp3,0e05e70f2e4943f8b8dd863380a55be0e34a9d35d0aaa1...,Domnia sa poate să-şi apere drepturile sale pr...,NaN,2,0,twenties,male_masculine,NaN,NaN,ro,NaN
3,07b9e73fe1516d96e2ced90d93c5ed7f9677b27ac2f332...,common_voice_ro_38390317.mp3,de8874a558645146354e4c4556a21581e71198890e219f...,"Avem, însă, și o responsabilitate europeană.",NaN,2,0,NaN,NaN,NaN,NaN,ro,NaN
4,09a39716d3b52346f9657fb78bb5daa0a7d5cff3ac4eb2...,common_voice_ro_27574161.mp3,2833d24c6304dd9f05b10ccb794483dea2cf305da993f9...,Avem într-adevăr multă libertate de acțiune.,NaN,2,0,fourties,male_masculine,NaN,NaN,ro,NaN


In [ ]:
from mutagen.mp3 import MP3
from collections import defaultdict
import hashlib
import shutil

clips_dir = metadata_path.parent / "clips"

df = df.dropna(
    subset=["client_id", "sentence", "path"]
).copy()

df = df[
    (df["client_id"].str.strip() != "") &
    (df["sentence"].str.strip() != "")
].copy()

df["word_count"] = (
    df["sentence"].str.split().str.len()
)

df = df[df["word_count"].between(5, 15)]
df = df.sample(frac=1, random_state=42)

selected = []
speaker_counts = defaultdict(int)
seen_sentences = set()

for _, row in df.iterrows():

    speaker = row["client_id"]
    sentence = row["sentence"].strip()

    if speaker_counts[speaker] >= 5:
        continue

    if sentence.casefold() in seen_sentences:
        continue

    audio_path = clips_dir / row["path"]

    if not audio_path.exists():
        continue

    try:
        duration = MP3(str(audio_path)).info.length
    except Exception:
        continue

    if not 3 <= duration <= 10:
        continue

    clip_id = f"ro_{len(selected)+1:03d}"

    destination = ROOT / "pilot/ro" / f"{clip_id}.mp3"

    shutil.copy2(audio_path, destination)

    selected.append({
        "clip_id": clip_id,
        "language": "ro",
        "speaker_id": hashlib.sha256(
            speaker.encode("utf-8")
        ).hexdigest()[:16],
        "text": sentence,
        "duration": round(duration, 3),
        "filename": destination.name,
        "source_path": row["path"],
        "selection_seed": 42
    })

    speaker_counts[speaker] += 1
    seen_sentences.add(sentence.casefold())

    if len(selected) == 50:
        break

pilot_df = pd.DataFrame(selected)

pilot_df.to_csv(
    ROOT / "metadata/ro_pilot.csv",
    index=False
)

print("Selected recordings:", len(pilot_df))
print("Unique speakers:", pilot_df["speaker_id"].nunique())

display(pilot_df.head())

Selected recordings: 50
Unique speakers: 37


,clip_id,language,speaker_id,text,duration,filename,source_path,selection_seed
0,ro_001,ro,96e61700b10000d6,Nu este pregătit să aibă o relație durabilă.,4.356,ro_001.mp3,common_voice_ro_26224879.mp3,42
1,ro_002,ro,4cd0522051ad99be,"Atunci, care sunt cele mai mari provocări?",3.696,ro_002.mp3,common_voice_ro_22357118.mp3,42
2,ro_003,ro,6b8e2c7aa0ab00ba,"Ceea ce este, de fapt, un lucru bun.",4.140,ro_003.mp3,common_voice_ro_25895761.mp3,42
3,ro_004,ro,4f12ab8016c7e128,Aceste forme de concurenţă trebuie oprite.,3.024,ro_004.mp3,common_voice_ro_21357595.mp3,42
4,ro_005,ro,0d7083189bed97ed,"Dacă nu, nu vom fi luaţi în serios.",4.896,ro_005.mp3,common_voice_ro_20367322.mp3,42


In [ ]:
from IPython.display import Audio, display

sample = pilot_df.iloc[13]

print("Transcript:", sample["text"])
print("Speaker:", sample["speaker_id"])

audio_path = ROOT / "pilot/ro" / sample["filename"]

display(Audio(filename=str(audio_path)))

Transcript: Să nu ai niciodată încredere în acela care şi-a încălcat o dată cuvântul.
Speaker: 5c6698926e577f69
